## Tool-Runner (SDK) for User-Defined Functions and Tools

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Defining the User-Defined Functions

In [ ]:
import json
import requests
from anthropic import beta_tool

@beta_tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location.
    
    Args:
        location: The city whose weather details need to be fetched
    """

    response = requests.get(
            f"https://wttr.in/{location}",
            params={
                "format": "j1"
            }
        )
    
    response.raise_for_status()
    
    return str(response.json())


@beta_tool
def calculate_sum(a: int, b: int) -> str:
    """Add two numbers together.

    Args:
        a: First number
        b: Second number
    """
    return str(a + b)

### Implement the Tool Runner with Anthropic Client

In [ ]:
runner = client.beta.messages.tool_runner(
    model = claude_model_name,
    max_tokens = 4096,
    tools = [get_weather, calculate_sum],
    messages = [
        {
            "role": "user",
            "content": "What's the weather like in London? Also, what's 15 + 27"
        }
    ]
)

for message in runner:
    if message.role == "assistant":

        for block in message.content:

            if block.type == "text":

                print("\n Assistant")
                print("-"*80)
                print(block.text)

            elif block.type == "tool_use":
                print("\nTool Call")
                print("-" * 80)

                print(f"Tool : {block.name}")

                print("Arguments:")

                for key, value in block.input.items():
                    print(f"  {key}: {value}")

    print("\n" + "=" * 80 + "\n")